# Logging & Defensive Programming

---

In production, you can't `print()` your way out of bugs. You need **structured, configurable, persistent** visibility into what your code is doing — and you need it to keep working even when inputs are bad.

## Why `print()` is Not Enough

- Can't be turned off without editing code
- No timestamps, no source context
- No severity levels (info vs error look the same)
- Goes to stdout — mixed with real program output
- No way to route to file, network, or syslog

---
## The `logging` Module

Python's built-in `logging` module solves all of these. Basic usage:

In [ ]:
import logging

logging.basicConfig(level=logging.INFO)

logging.debug("diagnostic detail")     # silent (below INFO)
logging.info("routine event")
logging.warning("something unexpected")
logging.error("operation failed")
logging.critical("app cannot continue")

> **Note:** The default level is `WARNING`, so `debug()` and `info()` are silent unless you call `basicConfig(level=...)` first.

---
## Log Levels Reference

| Level | When to use | Example |
|-------|-------------|---------|
| `DEBUG` | Diagnostic detail — verbose | `logger.debug(f"payload={data}")` |
| `INFO` | Routine events | `"service started"`, `"request processed"` |
| `WARNING` | Unexpected but recoverable | `"retrying connection"`, `"deprecated API used"` |
| `ERROR` | Operation failed, app continues | `"could not save order"` |
| `CRITICAL` | App cannot continue | `"database unreachable, shutting down"` |

---
## Formatting Log Messages

Add timestamps, level, and logger name with a format string:

```python
logging.basicConfig(
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    level=logging.INFO
)
```

**Common format specifiers:**

| Specifier | Meaning |
|-----------|---------|
| `%(asctime)s` | Timestamp |
| `%(levelname)s` | DEBUG / INFO / WARNING / ... |
| `%(name)s` | Logger name |
| `%(message)s` | The log message |
| `%(filename)s` | Source file |
| `%(lineno)d` | Line number |

In [ ]:
import logging
import importlib
importlib.reload(logging)   # reset in notebooks so basicConfig takes effect

logging.basicConfig(
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    level=logging.INFO
)

logging.info("Service started")
logging.warning("Cache miss")

---
## Module-Level Loggers (Best Practice)

Instead of using the root logger via `logging.info(...)`, create a **named logger** per module:

```python
logger = logging.getLogger(__name__)
logger.info("something happened")
```

**Why?** Loggers are hierarchical (`myapp.db`, `myapp.api`, ...). You can configure verbosity per module — turn DEBUG on just for `myapp.db` while everything else stays at INFO.

In [ ]:
logger = logging.getLogger("myapp.orders")
logger.info("Order module initialized")

---
## Logging to a File

Write logs to disk instead of stdout:

```python
logging.basicConfig(
    filename='app.log',
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s'
)
```

For more control (multiple destinations, rotation, etc.) use a `FileHandler`:

```python
handler = logging.FileHandler('app.log')
handler.setLevel(logging.INFO)
logger.addHandler(handler)
```

---
## Logging Exceptions

Inside an `except` block, use `logger.exception(...)` — it logs the message **plus the full traceback** automatically:

```python
try:
    1 / 0
except ZeroDivisionError:
    logger.exception("Math broke")
```

Use `logger.exception` only inside `except` blocks. Elsewhere, use `logger.error`.

In [ ]:
logger = logging.getLogger("demo")

try:
    result = 10 / 0
except ZeroDivisionError:
    logger.exception("Failed to divide")

---
# Defensive Programming
---

**Defensive programming** is the discipline of writing code that survives bad inputs and unexpected conditions.

Three core ideas:
1. **Validate inputs at the boundaries** of your system
2. **Assume external data is hostile** — never trust it blindly
3. **Fail fast and explicitly** — surface problems where they happen, not three layers deep

## Input Validation

- **Validate at the boundary** — where data enters your system (user input, API responses, file reads, env vars).
- **Trust internal code** — don't re-validate the same value at every function call.
- Raise a clear exception when validation fails.

In [ ]:
def validate_age(age):
    if not isinstance(age, int):
        raise TypeError(f"age must be int, got {type(age).__name__}")
    if age < 0 or age > 150:
        raise ValueError(f"age must be 0-150, got {age}")
    return age

print(validate_age(30))

try:
    validate_age("thirty")
except TypeError as e:
    print(f"Caught: {e}")

---
## `assert` vs `raise`

| | `assert` | `raise` |
|---|----------|---------|
| Purpose | Invariants / dev-time sanity checks | Runtime errors you expect to handle |
| Disabled by `python -O`? | **Yes** | No |
| Use for user input? | **Never** | Yes |
| Use for "this should never happen"? | Yes | Yes (with a specific exception) |

> **Rule of thumb:** if a user could trigger it, use `raise`. If only a programmer bug could trigger it, `assert` is okay.

In [ ]:
# assert — internal invariant
def average(numbers):
    assert len(numbers) > 0, "average() requires non-empty list"
    return sum(numbers) / len(numbers)

# raise — user-facing validation
def set_volume(level):
    if not 0 <= level <= 100:
        raise ValueError(f"volume must be 0-100, got {level}")
    return level

print(average([1, 2, 3]))
print(set_volume(75))

---
## Guard Clauses & Early Returns

Deeply nested `if/else` is hard to read. **Guard clauses** flip the conditions and return early on bad cases:

In [ ]:
# Bad — nested
def can_post_bad(user):
    if user is not None:
        if user.get("active"):
            if user.get("role") == "admin":
                return True
            else:
                return False
        else:
            return False
    else:
        return False

# Good — guard clauses, flat
def can_post(user):
    if user is None:
        return False
    if not user.get("active"):
        return False
    if user.get("role") != "admin":
        return False
    return True

print(can_post({"active": True, "role": "admin"}))
print(can_post({"active": False, "role": "admin"}))
print(can_post(None))

---
## Fail Fast Principle

- **Validate inputs at the top of functions** — don't let bad data flow deeper
- **Throw specific, helpful errors** — say what was wrong and what was expected
- **Don't silently swallow exceptions**

```python
# Code smell — hides everything
try:
    do_thing()
except:
    pass

# Better — catch what you expect, log what you don't
try:
    do_thing()
except ValueError as e:
    logger.warning(f"bad input: {e}")
```

---
## EAFP vs LBYL

Two styles for handling "might this fail?":

- **EAFP** — *Easier to Ask Forgiveness than Permission*: just do it, catch the exception. **Pythonic.**
- **LBYL** — *Look Before You Leap*: check conditions first, then act.

In [ ]:
d = {"name": "Alice"}

# LBYL
if "age" in d:
    age = d["age"]
else:
    age = 0
print("LBYL:", age)

# EAFP
try:
    age = d["age"]
except KeyError:
    age = 0
print("EAFP:", age)

# Most Pythonic for this specific case
age = d.get("age", 0)
print("get:", age)

---
## Common Defensive Patterns

- **Default values:** `dict.get(key, default)` instead of `d[key]` when key may be missing
- **`isinstance()` checks at boundaries only** — not deep in internal logic
- **Context managers (`with`)** for files, locks, connections — guarantees cleanup
- **Specific `except` clauses** — `except ValueError` not bare `except:`

---
# 📝 Assignments – Logging & Defensive Programming
---

## Task 1 – Replace `print()` with `logging`

**Problem:** Convert this debug-by-print function to use `logging` with proper levels:

```python
def process_order(order_id):
    print("starting")
    print("validated")
    print("saved")
```

Use `logging.INFO` for each step. Configure logging so INFO messages appear.

**Expected output (something like):**
```
INFO:root:starting order 42
INFO:root:validated order 42
INFO:root:saved order 42
```

**💡 Hint:** `logging.basicConfig(level=logging.INFO)` then `logging.info(f"...")`

In [ ]:
# Task 1 – process_order with logging
import logging

def process_order(order_id):
    pass

process_order(42)


## Task 2 – Log to a File with Timestamps

**Problem:** Configure logging to write to `app.log` with timestamps. Log one INFO and one ERROR message. Then read `app.log` and print its contents to verify.

**Expected output (something like):**
```
2026-06-23 10:00:00 [INFO] App started
2026-06-23 10:00:00 [ERROR] Something failed
```

**💡 Hint:** `logging.basicConfig(filename='app.log', format='%(asctime)s [%(levelname)s] %(message)s', level=logging.INFO)`. In notebooks you may need `importlib.reload(logging)` first.

In [ ]:
# Task 2 – Log to file
import logging, importlib
importlib.reload(logging)

# configure logging here

# log one INFO and one ERROR

# read app.log and print its contents


## Task 3 – `divide(a, b)` with Guard Clauses

**Problem:** Write `divide(a, b)` that:
- Raises `TypeError` if either argument isn't `int` or `float`
- Raises `ZeroDivisionError` if `b == 0`
- Otherwise returns `a / b`

Then call it with valid and invalid inputs, catch the exceptions, and **log** them.

**Expected output (something like):**
```
5.0
ERROR:root:Cannot divide by zero
ERROR:root:Inputs must be numeric
```

**💡 Hint:** Use `isinstance(x, (int, float))` for the type check. Use `logger.exception(...)` inside `except`.

In [ ]:
# Task 3 – divide with guard clauses
import logging
logger = logging.getLogger(__name__)

def divide(a, b):
    pass

for a, b in [(10, 2), (10, 0), ("x", 2)]:
    try:
        print(divide(a, b))
    except Exception:
        logger.exception(f"divide({a!r}, {b!r}) failed")


## Task 4 – Refactor Nested if/else to Guard Clauses

**Problem:** Refactor this nested function using guard clauses (early returns):

```python
def is_admin(user):
    if user is not None:
        if user.get("is_active"):
            if user.get("role") == "admin":
                return True
            else:
                return False
        else:
            return False
    else:
        return False
```

**Expected output:**
```
True
False
False
False
```

**💡 Hint:** Flip each condition and `return False` early; only the success path stays at the end.

In [ ]:
# Task 4 – Refactor to guard clauses
def is_admin(user):
    pass

print(is_admin({"is_active": True, "role": "admin"}))
print(is_admin({"is_active": True, "role": "user"}))
print(is_admin({"is_active": False, "role": "admin"}))
print(is_admin(None))


## 🚀 Bonus – `divide_safely(a, b)`

**Problem:** Build `divide_safely(a, b)` that:
- Validates inputs (numeric, non-zero divisor)
- Logs each attempt with `logger.info`
- On failure, logs with `logger.exception` and returns `None` (does not raise)
- On success, returns `a / b`

Test with: `(10, 2)`, `(10, 0)`, `("x", 5)`, `(7, 3)`.

**Expected output (something like):**
```
5.0
None
None
2.333...
```

**💡 Hint:** Wrap the `divide` logic in `try/except Exception`; in the except block call `logger.exception(...)` and `return None`.

In [ ]:
# Bonus – divide_safely
import logging
logger = logging.getLogger(__name__)

def divide_safely(a, b):
    pass

for a, b in [(10, 2), (10, 0), ("x", 5), (7, 3)]:
    print(divide_safely(a, b))
